# 1. Imports

In [51]:
import numpy as np
import pandas as pd

# 2. Load datasets

In [52]:
sheet_name1 = ['900069',"900080",'900103','900104','900106','900110','900113',"900119",'900120','900128']
df1 = pd.read_excel("Dataset/1.xlsx", sheet_name=sheet_name1)
sheet_name2 = ['900129','900130','900133','900136','900140','900143','900150','900160','900177','900182']
df2 = pd.read_excel("Dataset/2.xlsx", sheet_name=sheet_name2)
sheet_name3 = ['900184',"900190",'900193','900198','900207','900208','900210',"900212",'900213','900214','900253']
df3 = pd.read_excel("Dataset/3.xlsx", sheet_name=sheet_name3)
sheet_name4 = ['900216','900217','900218','900222','900223','900226','900228','900231','900234','900246','900249','900252']
df4 = pd.read_excel("Dataset/4.xlsx", sheet_name=sheet_name4)

In [53]:
for sheet in sheet_name1:
    df1[sheet] = df1[sheet].sort_values(by='Load Survey Time', ascending=True)
for sheet in sheet_name2:
    df2[sheet] = df2[sheet].sort_values(by='Load Survey Time', ascending=True)
for sheet in sheet_name3:
    df3[sheet] = df3[sheet].sort_values(by='Load Survey Time', ascending=True)
for sheet in sheet_name4:
    df4[sheet] = df4[sheet].sort_values(by='Load Survey Time', ascending=True)
concat_df1 = pd.concat(df1.values(), ignore_index=True)
concat_df2 = pd.concat(df2.values(), ignore_index=True)
concat_df3 = pd.concat(df3.values(), ignore_index=True)
concat_df4 = pd.concat(df4.values(), ignore_index=True)
concat_df = pd.concat([concat_df1, concat_df2, concat_df3, concat_df4], ignore_index=True)

# 3. Pre-Processing

## 3.1 Handle Duplicates

In [54]:
print("Duplicate rows:", concat_df.duplicated().sum())

print("Duplicate meter-timestamp pairs:",
      concat_df.duplicated(subset=["MID", "Load Survey Time"]).sum())

Duplicate rows: 16
Duplicate meter-timestamp pairs: 16


In [55]:
duplicates = concat_df[concat_df.duplicated(subset=["MID", "Load Survey Time"],keep=False)].sort_values(["MID", "Load Survey Time"])
duplicates

,MID,Load Survey Time,R Phase Current,Y Phase Current,B Phase Current,R Phase Voltage,Y Phase Voltage,B Phase Voltage,Active Energy,Apparent Energy,Reactive Lag,Reactive Lead,PF
156646,900106,2017-01-16 00:00:00,0.3,8.33,10.53,256.5,255.7,256.7,1.23,1.23,0.00,0.10,-1.000
156647,900106,2017-01-16 00:00:00,0.3,8.33,10.53,256.5,255.7,256.7,1.23,1.23,0.00,0.10,-1.000
156648,900106,2017-01-16 00:15:00,0.3,8.31,10.42,256.8,255.9,256.8,1.21,1.21,0.00,0.10,-1.000
156649,900106,2017-01-16 00:15:00,0.3,8.31,10.42,256.8,255.9,256.8,1.21,1.21,0.00,0.10,-1.000
156650,900106,2017-01-16 00:30:00,0.3,7.95,10.46,257.4,256.6,257.2,1.20,1.21,0.00,0.09,-0.992
156651,900106,2017-01-16 00:30:00,0.3,7.95,10.46,257.4,256.6,257.2,1.20,1.21,0.00,0.09,-0.992
156652,900106,2017-01-16 00:45:00,0.3,7.83,10.15,257.3,256.7,257.4,1.18,1.18,0.00,0.10,-1.000
156653,900106,2017-01-16 00:45:00,0.3,7.83,10.15,257.3,256.7,257.4,1.18,1.18,0.00,0.10,-1.000
156654,900106,2017-01-16 01:00:00,0.3,8.05,10.08,258.0,257.2,257.6,1.18,1.18,0.00,0.10,-1.000
156655,900106,2017-01-16 01:00:00,0.3,8.05,10.08,258.0,257.2,257.6,1.18,1.18,0.00,0.10,-1.000


In [56]:
df = concat_df.drop_duplicates()

## 3.2 Handle missing timestamps 

In [57]:
df["time_diff"] = (
    df.groupby("MID")["Load Survey Time"]
      .diff()
)
gaps = df[df["time_diff"] > pd.Timedelta(minutes=15)]
gaps

,MID,Load Survey Time,R Phase Current,Y Phase Current,B Phase Current,R Phase Voltage,Y Phase Voltage,B Phase Voltage,Active Energy,Apparent Energy,Reactive Lag,Reactive Lead,PF,time_diff
16032,900069,2017-06-21,48.12,48.01,0.00,248.9,237.3,243.2,5.02,5.54,1.65,1.31,0.906,4 days 00:15:00
129182,900104,2017-05-24,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,35 days 00:15:00
158862,900106,2017-02-09,0.75,9.17,11.34,252.6,252.2,253.4,1.31,1.34,0.11,0.08,0.978,1 days 00:15:00
171150,900106,2017-06-20,0.40,8.66,11.36,245.2,246.9,242.5,1.24,1.25,0.00,0.07,-0.992,3 days 00:15:00
397578,900129,2017-06-28,10.73,0.00,10.76,213.4,220.0,218.7,1.01,1.08,0.29,0.30,-0.935,79 days 21:00:00
403722,900129,2017-09-01,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,1 days 00:15:00
436288,900130,2017-06-20,1.04,0.98,2.88,254.5,255.1,249.2,0.16,0.32,0.02,0.19,-0.500,2 days 23:45:00
588676,900143,2017-04-12,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,4 days 00:15:00
588772,900143,2017-07-12,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,90 days 00:15:00
945019,900207,2017-06-20,0.40,0.09,0.00,265.3,267.7,269.0,0.04,0.04,0.00,0.01,-1.000,3 days 00:15:00


In [58]:
def regularize_meter(group, mid):

    group = group.sort_values("Load Survey Time").copy()

    # Make sure timestamp is datetime
    group["Load Survey Time"] = pd.to_datetime(
        group["Load Survey Time"]
    )

    # Remove MID temporarily because it is the group identifier
    group = group.drop(columns=["MID"], errors="ignore")

    # Set timestamp as index
    group = group.set_index("Load Survey Time")

    # Create 15-minute timeline
    full_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="15min"
    )

    # Reindex
    group = group.reindex(full_index)

    group.index.name = "Load Survey Time"

    # Restore MID
    group["MID"] = mid

    return group.reset_index()


df = pd.concat(
    [
        regularize_meter(group, mid)
        for mid, group in df.groupby("MID")
    ],
    ignore_index=True
)

In [59]:
df

,Load Survey Time,R Phase Current,Y Phase Current,B Phase Current,R Phase Voltage,Y Phase Voltage,B Phase Voltage,Active Energy,Apparent Energy,Reactive Lag,Reactive Lead,PF,time_diff,MID
0,2017-01-01 00:00:00,34.38,34.30,0.00,261.0,255.4,0.0,1.20,1.29,0.34,0.34,0.930,NaT,900069
1,2017-01-01 00:15:00,42.93,42.83,0.00,260.3,253.4,0.0,1.09,1.18,0.32,0.32,0.924,0 days 00:15:00,900069
2,2017-01-01 00:30:00,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,0 days 00:15:00,900069
3,2017-01-01 00:45:00,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,0 days 00:15:00,900069
4,2017-01-01 01:00:00,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,0 days 00:15:00,900069
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707286,2018-02-20 00:45:00,1.30,1.82,3.40,262.6,260.8,262.8,0.25,0.44,0.33,0.00,0.568,0 days 00:15:00,900253
1707287,2018-02-20 01:00:00,1.31,1.20,3.47,263.3,261.8,263.6,0.21,0.41,0.35,0.00,0.512,0 days 00:15:00,900253
1707288,2018-02-20 01:15:00,1.30,1.20,3.46,263.4,262.0,263.8,0.21,0.41,0.35,0.00,0.512,0 days 00:15:00,900253
1707289,2018-02-20 01:30:00,0.76,0.48,2.68,263.2,261.9,263.6,0.15,0.27,0.21,0.01,0.556,0 days 00:15:00,900253


# 4. Saving the Clean Data

In [69]:
column_order = [
    'MID',
    'Load Survey Time',
    'R Phase Current',
    'Y Phase Current',
    'B Phase Current',
    'R Phase Voltage',
    'Y Phase Voltage',
    'B Phase Voltage',
    'Active Energy',
    'Apparent Energy',
    'Reactive Lag',
    'Reactive Lead',
    'PF',
    'time_diff'
]

df = df[column_order]
df.to_csv("Dataset/cleaned_data.csv", index=False)